# 第 7 章 · 多 agent（LoopAgent）

一个干活的 agent + 一个检查状态的自定义 agent，循环到**条件满足**或撞 `max_iterations`。

⚠️ 书里那版永远停不下来：instruction 让模型"把 status 设成 completed"，
**但 LlmAgent 写不了 state**——它只能"说"完成了，状态纹丝不动。下面用工具补上。

★ 本版把"完成任务"做成**随机事件**（每轮 `DONE_PROB` 概率成功），所以能真正看到
循环跑多轮、以及"条件停"和"预算停"的区别。

★ 结构是"上层 agent 总结下层 loop"：

```
Supervisor（上层 LLM agent）
   └─ AgentTool(loop)   ← 把整个 LoopAgent 当工具调用
        └─ LoopAgent [Worker, ConditionChecker]
```

数据走 **function_response**：`ConditionChecker` 在跳出循环那一轮把统计文本作为自己的
输出交出去 → 成为 loop 的"返回值" → 上层 agent 拿到后写总结。和 AgentTool 那份同一机制，
只是被包装的对象从"一个 agent"变成了"一个完整的循环"。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break

# 通用脚手架都在 common 里：run_once / run_session / run_async / 重试 / 静音
from common import (adk_model, quiet_adk_sync_warnings, retry_call,
                    run_once, run_session, run_async, require_adk)
quiet_adk_sync_warnings()

require_adk()      # 【适配】内核选错时给出可操作提示（内部就是 import google.adk）

import uuid
from google.adk.runners import InMemoryRunner


import json
import random
from typing import AsyncGenerator

from google.adk.agents import BaseAgent, LlmAgent, LoopAgent
from google.adk.agents.invocation_context import InvocationContext
from google.adk.events import Event, EventActions
from google.adk.tools import ToolContext, agent_tool
from google.genai import types

DONE_PROB = 0.4          # ★ 每轮"完成任务"的概率；调低能看到更多轮


def try_finish(tool_context: ToolContext) -> str:
    """尝试完成当前步骤。成功返回 DONE 并把 status 置为 completed；否则返回 NOT_DONE。"""
    # 【适配】书里让模型直接改 state —— LlmAgent 做不到，只能通过工具写
    n = tool_context.state.get("attempts", 0) + 1
    tool_context.state["attempts"] = n
    ok = random.random() < DONE_PROB
    tool_context.state.setdefault("log", []).append(
        f"try#{n} {'DONE' if ok else 'NOT_DONE'}")
    if ok:
        tool_context.state["status"] = "completed"
        return "DONE"
    return "NOT_DONE: 本轮没做成，还有问题没解决"


class ConditionChecker(BaseAgent):
    """读 state 判断是否结束；结束那一轮把【统计】作为自己的输出交出去。"""
    name: str = "ConditionChecker"
    description: str = "Checks if the loop is complete; on completion it also reports the stats."

    async def _run_async_impl(self, context: InvocationContext) -> AsyncGenerator[Event, None]:
        s = context.session.state
        if s.get("status") == "completed":
            # ★ 关键：escalate 的同时带上 content —— 这段文本就是整个 loop 的"返回值"，
            #   上层 agent 通过 AgentTool 拿到的就是它。
            stats = (f"轮次统计: 尝试={s.get('attempts', 0)} 次; "
                     f"最终状态=completed; 逐轮={s.get('log')}")
            yield Event(author=self.name,
                        content=types.Content(role="model", parts=[types.Part(text=stats)]),
                        actions=EventActions(escalate=True))
        else:
            yield Event(author=self.name,
                        content=types.Content(role="model", parts=[types.Part(text="继续下一轮")]))


worker = LlmAgent(
    name="Worker",
    model=adk_model(),                      # 【适配】gemini-2.0-flash-exp 已下线
    instruction="每一轮你必须先调用 try_finish 工具（这是你唯一的动作），"
                "再根据返回值说一句话：DONE 就说“已完成”，否则说“还需要继续”。"
                "不要只说话而不调用工具。",
    tools=[try_finish],
)

loop = LoopAgent(name="Poller", max_iterations=6, sub_agents=[worker, ConditionChecker()])

# ★ 上层 agent：把整个 loop 当【工具】调用，拿到它的返回值（含轮次统计）后再总结。
#   数据走的是 function_response —— 和 AgentTool 那份是同一个机制，
#   只是这里被包装的"工具"是一个完整的 LoopAgent。
supervisor = LlmAgent(
    name="Supervisor",
    model=adk_model(),
    instruction="调用 Poller 工具执行循环任务。它返回的文本里带有轮次统计，"
                "请用两三句中文总结：跑了几轮、每轮结果如何、最终是否成功。",
    tools=[agent_tool.AgentTool(agent=loop)],
)


def run_supervised(seed):
    random.seed(seed)                       # 固定随机性，方便复现
    runner = InMemoryRunner(supervisor)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    report = ""
    for ev in runner.run(user_id="u", session_id=sid,
                         new_message=types.Content(role="user", parts=[types.Part(text="跑循环并汇报")])):
        if not ev.content:
            continue
        for p in (ev.content.parts or []):
            if getattr(p, "function_response", None):
                print(f"  [下层 loop 的返回值] {str(p.function_response.response)[:200]}")
            elif getattr(p, "text", None) and p.text.strip() and getattr(ev, "author", "") == "Supervisor":
                report = p.text.strip()
    print(f"  [上层 agent 的总结] {report}")


for seed in (2, 4):
    print(f"\n===== seed={seed} =====")
    run_supervised(seed)



===== seed=2 =====


  [下层 loop 的返回值] {'result': "轮次统计: 尝试=3 次; 最终状态=completed; 逐轮=['try#1 NOT_DONE', 'try#2 NOT_DONE', 'try#3 DONE']"}


  [上层 agent 的总结] 完成，第三轮成功完成，最终任务圆满结束。

===== seed=4 =====


  [下层 loop 的返回值] {'result': "轮次统计: 尝试=1 次; 最终状态=completed; 逐轮=['try#1 DONE']"}


  [上层 agent 的总结] 本次循环任务共执行了 1 轮。在第一轮尝试中，任务便顺利完成（结果为 'try#1 DONE'）。最终，整个轮询任务已成功结束。
